# Import Required Libraries
Import PySpark, pandas, and other necessary libraries for data processing and visualization.

In [1]:
# Import Required Libraries
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Configure Spark Session for ClickHouse
Set up Spark session builder with required ClickHouse connector packages.

In [2]:
# Configure Spark Session for ClickHouse
packages = [
    "com.clickhouse.spark:clickhouse-spark-runtime-3.5_2.12:0.8.1",
    "com.clickhouse:clickhouse-client:0.9.4",
    "com.clickhouse:clickhouse-http-client:0.9.4",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1"
]


# Set ClickHouse Connection Parameters
Define ClickHouse host, port, database, user, and password variables.

In [3]:
# Set ClickHouse Connection Parameters
clickhouse_host = "localhost"
clickhouse_port = 9000
clickhouse_http_port = 8123
clickhouse_database = "public"
clickhouse_user = "default"
clickhouse_password = "DfsTeChB1"
table_name = "stixor_fraud_features_distributed"
selected_features = [
    'cutoff_date', 'fraud_flag', 'trx_channel', 'trx_type', 
    'start_balance', 'trx_amt', 'mbar_registered_channel',
    'hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 
    'is_business_hours', 'is_unusual_hour', 'night_weekend_combo',
    'txn_txns_3d', 'txn_total_amount_3d', 'txn_avg_amount_3d',
    'txn_max_amount_3d', 'txn_min_amount_3d', 'txn_unique_recipients_3d',
    'txn_unique_channels_3d', 'txn_unique_types_3d', 'txn_is_high_activity_3d',
    'txn_multi_channel_recent', 'txn_amount_deviation_from_avg',
    'txn_night_txns_3d', 'txn_weekend_txns_3d',
    'channel_new_jc_app', 'channel_ussd', 'channel_ussd_api',
    'channel_payment_gateway', 'channel_mobile_app',
    'type_transfer_c2c', 'type_transfer_c2b', 'type_bill_payment',
    'type_mobile_load', 'user_total_txns_3d', 'user_total_amount_3d',
    'user_avg_amount_3d', 'user_max_amount_3d', 'user_unique_recipients_3d',
    'user_unique_channels_3d', 'user_total_txns_7d', 'user_avg_amount_7d',
    'user_max_amount_7d', 'user_night_txns_7d', 'user_weekend_txns_7d'
]

# Initialize Spark Session
Create and initialize the Spark session with the ClickHouse catalog configuration.

In [ ]:
# Initialize Spark Session
packages = [
    "com.clickhouse.spark:clickhouse-spark-runtime-3.5_2.12:0.8.1",
    "com.clickhouse:clickhouse-client:0.9.4",
    "com.clickhouse:clickhouse-http-client:0.9.4",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1"
]
spark = (SparkSession.builder
    .appName("spark-clickhouse-demo")
    .master("local[*]")
    .config("spark.jars.packages", ",".join(packages))
    .getOrCreate()
)
spark.conf.set("spark.sql.catalog.clickhouse", "com.clickhouse.spark.ClickHouseCatalog")
spark.conf.set("spark.sql.catalog.clickhouse.host", clickhouse_host)
spark.conf.set("spark.sql.catalog.clickhouse.protocol", "http")
spark.conf.set("spark.sql.catalog.clickhouse.http_port", str(clickhouse_http_port))
spark.conf.set("spark.sql.catalog.clickhouse.user", clickhouse_user)
spark.conf.set("spark.sql.catalog.clickhouse.password", clickhouse_password)
spark.conf.set("spark.sql.catalog.clickhouse.database", clickhouse_database)
spark.conf.set("spark.clickhouse.write.format", "json")

:: loading settings :: url = jar:file:/root/miniconda3/envs/fraud-spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.clickhouse.spark#clickhouse-spark-runtime-3.5_2.12 added as a dependency
com.clickhouse#clickhouse-client added as a dependency
com.clickhouse#clickhouse-http-client added as a dependency
org.apache.httpcomponents.client5#httpclient5 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4349cf21-8c36-42c2-aa41-b9667089f8bd;1.0
	confs: [default]
	found com.clickhouse.spark#clickhouse-spark-runtime-3.5_2.12;0.8.1 in central
	found com.clickhouse#clickhouse-client;0.9.4 in central
	found com.clickhouse#clickhouse-data;0.9.4 in central
	found commons-codec#commons-codec;1.17.1 in central
	found commons-io#commons-io;2.16.1 in central
	found org.apache.commons#commons-lang3;3.18.0 in central
	found org.apache.commons#commons-compress;1.27.1 in central
	found com.clickhouse#clickhouse-http-client;0.9.4 in central
	found org.apache.httpcomponents.client5#http

25/11/27 15:09:42 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


# Read Data from ClickHouse Table
Use Spark SQL to query the ClickHouse table and load the data into a Spark DataFrame.

In [7]:
clickhouse_url = f"jdbc:clickhouse://{clickhouse_host}:{clickhouse_port}/{clickhouse_database}"

In [8]:
# Query ClickHouse table and load data into Spark DataFrame
# Replace 'your_table_name' with the actual table name
clickhouse_table = "public.stixor_fraud_features_distributed"

# Example query: select all columns
query = f"SELECT * FROM {clickhouse_table} LIMIT 1000"

# Read data using Spark SQL
spark_df = spark.read \
    .format("jdbc") \
    .option("url", clickhouse_url) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("dbtable", f"({query}) as tmp") \
    .option("user", clickhouse_user) \
    .option("password", clickhouse_password) \
    .load()

# Show the first few rows
spark_df.show(5)

Py4JJavaError: An error occurred while calling o40.load.
: java.lang.ClassNotFoundException: com.clickhouse.jdbc.ClickHouseDriver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:593)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:526)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationP

# Display Data Sample and Class Distribution
Show a sample of the loaded data and analyze the class distribution for the target variable.

In [ ]:
# Show a sample of the data
spark_df.show(10)

# Display schema
spark_df.printSchema()

# Analyze class distribution (replace 'target' with your label column)
target_col = "target"  # Change to your actual target column name
spark_df.groupBy(target_col).count().show()